In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx

import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
from pyscf import gto, scf, fci
import time
from NES_VMC_H2_631G import get_ccsd_excitations_and_sampler_edges_from_hf
from NES_VMC import NESTotalAnsatz, create_machine,\
        SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,\
        compute_qgt,sampler_info,NESTotalAnsatz_stable,NES_loss_energy_stable,create_machine_stable
        
import logging
# ========== 你原有全局参数（直接复用） ==========
bond_length = 1.8
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot
print(f'HF 基准能量: {hf_ground_energy:.8f}')

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 3  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
ha = nkx.operator.from_pyscf_molecule(mol)
Hatree_Fock = hi.all_states()[0]
single_edges, singles, doubles = get_ccsd_excitations_and_sampler_edges_from_hf(
    Hatree_Fock
)
print(f'single_edges: {single_edges}')
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
#sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: Prefer the new nk.driver.VMC_SR over VMC which supports minSR and SPRING.

E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV
HF 基准能量: -0.94605220
H₂ FCI 基准能量
E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV
single_edges: [(3, 0), (3, 1), (3, 2), (7, 4), (7, 5), (7, 6)]


In [2]:
import jax
import jax.numpy as jnp
import netket as nk

SINGLE_SIZE = hi.size

@nk.utils.struct.dataclass
class NESFermionHopRule(nk.sampler.rules.MetropolisRule):
    edges: jnp.ndarray
    K: int = nk.utils.struct.static_field()
    single_size: int = nk.utils.struct.static_field()

    def _check_duplicate(self, sigma_ext):
        """NES约束：检测任意两个子组态重复
        兼容一维单样本(返回标量) / 二维批量(返回batch数组)
        """
        one_d_input = (sigma_ext.ndim == 1)
        if one_d_input:
            sigma_ext = sigma_ext[None, :]
        
        batch_dim = sigma_ext.shape[0]
        sub = sigma_ext.reshape((batch_dim, self.K, self.single_size))
        # 全部子组态两两比对
        pair_equal = jnp.all(sub[:, :, None, :] == sub[:, None, :], axis=-1)
        diag_mask = jnp.eye(self.K, dtype=jnp.bool_)[None, :, :]
        off_diag_dup = jnp.where(diag_mask, False, pair_equal)
        batch_dup = jnp.any(off_diag_dup, axis=(-2, -1))
        
        if one_d_input:
            return batch_dup.squeeze()
        return batch_dup

    # 修复：补齐完整7个形参：self, sampler, machine, parameters, state, rng, sigma
    def transition(self, sampler, machine, parameters, state, rng, sigma):
        """跃迁规则"""
        batch_size = sigma.shape[0]
        key1, key2 = jax.random.split(rng)

        e_idx = jax.random.randint(key1, (batch_size,), 0, self.edges.shape[0])
        sel_e = self.edges[e_idx]
        i, j = sel_e[:,0], sel_e[:,1]

        sigma_cand = sigma.at[jnp.arange(batch_size),i].set(sigma[jnp.arange(batch_size),j])
        sigma_cand = sigma_cand.at[jnp.arange(batch_size),j].set(sigma[jnp.arange(batch_size),i])

        invalid = self._check_duplicate(sigma_cand)
        new_sigma = jnp.where(invalid[:, None], sigma, sigma_cand)

        return new_sigma, None

    def random_state(self, sampler, machine, parameters, state, rng):
        """随机态生成（完全不变）"""
        sigma_shape = state.σ.shape
        hilbert = sampler.hilbert

        def gen_single(key):
            max_tries = 100
            def cond(c): 
                return (c[0] < max_tries) & c[2]
            
            def body(c):
                tries, k, _, _ = c
                k, k_new = jax.random.split(k)
                s = hilbert.random_state(k_new)
                is_dup = self._check_duplicate(s)  # 一维输入自动返回标量
                return (tries + 1, k, is_dup, s)
            
            # 初始值 c[2] = True（标量布尔值），匹配while_loop
            init_c = (0, key, True, hilbert.random_state(key))
            final_c = jax.lax.while_loop(cond, body, init_c)
            tries, _, is_dup, s = final_c
            return jax.lax.cond(is_dup, lambda: hilbert.random_state(key), lambda: s)
        
        keys = jax.random.split(rng, sigma_shape[0])
        return jax.vmap(gen_single)(keys)

In [ ]:
hi.all_states()[0]

In [3]:
SINGLE_SIZE = hi.size
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）
ext_edges

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)

In [4]:
import logging
# 日志配置
logger = logging.getLogger('NES_VMC_K4')
logger.setLevel(logging.INFO)
# 阻止日志向上传播
logger.propagate = False
# 清除所有旧handler，防止重复打印
logger.handlers.clear()

# 自定义日志格式：只打印内容，不带等级、logger名
simple_formatter = logging.Formatter("%(message)s", datefmt="%H:%M:%S")

# 1. 文件输出处理器
file_handler = logging.FileHandler("nes_vmc_0625_K3.log", mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
console_handler.setLevel(logging.INFO)
logger.addHandler(console_handler)

print('库导入完成')

库导入完成


$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [7]:
import time
import jax
import jax.numpy as jnp
import optax

# ====================== 超参统一配置 ======================
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 400
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = True
clip_norm = 1.5        # 全局梯度L2上限，QML推荐1~2
lr = 0.01
qgt_diag_shift = 0.1  # 上调正则，抑制QGT梯度爆炸

# ====================== 模型初始化 ======================
total_ansatz = NESTotalAnsatz(SINGLE_SIZE, K, 12, rngs=nnx.Rngs(11))
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, _, _ = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, _, _ = create_single_machine(ansatz)
    single_machine_list.append(m)

# ====================== 优化器：梯度裁剪 + SGD ======================
# chain顺序：先裁剪梯度，再SGD更新
optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr)
)
opt_state = optimizer.init(total_params)

# ====================== 扩展采样边、自定义采样器 ======================
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE
)

# 适配二维采样输入的包装函数，修复init_shape报错
def machine_wrapper(params, sigma_2d):
    n_batch = sigma_2d.shape[0]
    sigma_3d = sigma_2d.reshape(n_batch, K, SINGLE_SIZE)
    return total_machine(params, sigma_3d)

sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(machine_wrapper, total_params, sampler_rng)

# ====================== 训练历史（新增裁剪后梯度监控） ======================
history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_3st': [],
    'loss': [],
    'params': [],
    'E_Lmatrix': [],
    'samples': [],
    'log_Psi_mean': [],
    'log_Psi_min': [],
    'log_Psi_max': [],
    'grad_norm_raw': [],       # QGT前原始梯度
    'grad_norm_natural': [],   # QGT自然梯度（裁剪前）
    'grad_norm_clipped': [],   # 裁剪后真实梯度（≤clip_norm）
}

logger.info("\n" + "="*60)
logger.info(f"开始多链 NES-VMC 训练 | 使用{'自然' if Natural_Grad else '原始'}梯度")
logger.info("="*60)
logger.info(f"FCI基准：基态={E_fcis[0]:.8f} Ha | 1激发={E_fcis[1]:.8f} Ha | 2激发={E_fcis[2]:.8f} Ha | 3激发={E_fcis[3]:.8f} Ha")
logger.info(f"理论 Loss 上限：{sum(E_fcis[0:3]):.8f} ")
logger.info(f"超参：clip_norm={clip_norm}, lr={lr}, QGT diag_shift={qgt_diag_shift}")

start_time = time.time()
for step in range(N_ITER):
    # 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=machine_wrapper,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)

    # 1. 原始变分梯度
    grad_raw, loss_mean, E_L_mean = nes_vmc_gradient(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch
    )
    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = jnp.linalg.norm(grad_raw_flat)
    grad_update = grad_raw

    # 2. QGT自然梯度预条件
    if Natural_Grad:
        qgt_reg_mat, _ = compute_qgt(
            total_machine, total_params, x_batch, diag_shift=qgt_diag_shift
        )
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        grad_update = unravel_fn(ng_flat)
        grad_norm_natural = jnp.linalg.norm(ng_flat)
    else:
        grad_norm_natural = grad_norm_raw

    # 3. 梯度裁剪（optimizer.update内部自动执行）
    updates, opt_state = optimizer.update(grad_update, opt_state, total_params)
    # 单独计算裁剪后梯度范数用于监控
    clip_transform = optax.clip_by_global_norm(clip_norm)
    clipped_grad, _ = clip_transform.update(grad_update, opt_state[0], total_params)
    clipped_flat, _ = ravel_pytree(clipped_grad)
    grad_norm_clipped = jnp.linalg.norm(clipped_flat)

    # 参数更新
    total_params = optax.apply_updates(total_params, updates)

    # 波函数输出
    log_Psi_batch = total_machine(total_params, x_batch)
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)

    # 记录历史
    history['step'].append(step)
    history['loss'].append(loss_mean)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['energy_3st'].append(eig_vals[3])
    history['params'].append(total_params)
    history['grad_norm_raw'].append(grad_norm_raw)
    history['grad_norm_natural'].append(grad_norm_natural)
    history['grad_norm_clipped'].append(grad_norm_clipped)

    # 打印日志，清晰区分三层梯度
    if step % 10 == 0 or step == N_ITER - 1:
        logger.info(f"[Step {step:3d}] logΨ: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")
        logger.info(f"梯度监控 | raw={grad_norm_raw:.4f} | natural={grad_norm_natural:.4f} | clipped={grad_norm_clipped:.4f}(上限{clip_norm})")
        logger.info(f"Loss={loss_mean:.6f} | E0={eig_vals[0]:.8f} | E1={eig_vals[1]:.8f} | E2={eig_vals[2]:.8f} | E3={eig_vals[3]:.8f}")
        logger.info("#-----------------------------------------#")

end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print("\n" + "="*60)
print("训练完成!")
print("="*60)


开始多链 NES-VMC 训练 | 使用自然梯度
FCI基准：基态=-1.02613572 Ha | 1激发=-0.97892204 Ha | 2激发=-0.66776157 Ha | 3激发=-0.60817046 Ha
理论 Loss 上限：-2.67281933 
超参：clip_norm=1.5, lr=0.01, QGT diag_shift=0.1
[Step   0] logΨ: mean=0.357+0.205j | min=-1.777-2.104j | max=1.169+0.049j
梯度监控 | raw=3.3473 | natural=16.7987 | clipped=1.5000(上限1.5)
Loss=0.686534 | E0=-0.79445453 | E1=0.04814401 | E2=1.43284447 | E3=1.43284447
#-----------------------------------------#
[Step  10] logΨ: mean=0.927+0.429j | min=-1.426-1.888j | max=1.913+1.733j
梯度监控 | raw=2.8338 | natural=16.0053 | clipped=1.5000(上限1.5)
Loss=0.013971 | E0=-0.77252212 | E1=-0.05765993 | E2=0.84415265 | E3=0.84415265
#-----------------------------------------#
[Step  20] logΨ: mean=2.844-0.013j | min=-1.376-1.497j | max=3.701+1.801j
梯度监控 | raw=1.3218 | natural=7.6080 | clipped=1.5000(上限1.5)
Loss=-0.392664 | E0=-1.05633729 | E1=-0.05378129 | E2=0.71745472 | E3=0.71745472
#-----------------------------------------#
[Step  30] logΨ: mean=3.325+0.145j | min=-1.

KeyboardInterrupt: 